# v2 — CAFEIN-aligned implementation

This notebook is a parallel **v2** of the original proof-of-concept. The original notebook is intentionally left unchanged.

**Goal:** reproduce the same research question while following the CAFEIN workflow documented in the environmental-exposure guide as closely as the public API allows.

Important distinction:
- CAFEIN directly defines route/network exposure through `StreetNetwork`, `Exposure`, and `DetailedItineraries`.
- Residential point exposure and OD destination-weighted exposure are not defined as native CAFEIN routing operations in the documentation supplied for this comparison. Those parts therefore remain analytical extensions around the CAFEIN sample data and are labelled as such.

All v2 outputs are written under `../outputs/v2/` so they can be compared with v1 without overwriting anything.

## 1. Load CAFEIN sample mobility and environmental data

This stage remains an **analytical extension** because the supplied CAFEIN exposure documentation does not define trip-weighted destination concentration as a native routing operation.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
import rioxarray

import cafein.sampledata.helsinki as helsinki

OUTPUT_DIR = Path("../outputs/v2/02_mobility_weighted_destination_exposure")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

od = pd.read_parquet(helsinki.mobility.od_weekday)
units = gpd.read_file(helsinki.mobility.neighbourhoods)
units["unit_id"] = units["Neighbourhood_unit_ID"].astype("Int64")
od["origin"] = od["origin"].astype("Int64")
od["destination"] = od["destination"].astype("Int64")

air_quality = rioxarray.open_rasterio(helsinki.air_quality, masked=True)
air_quality_bands = list(air_quality.attrs["long_name"])
pm25_band = air_quality_bands.index("PM25Concentration [ug/m3]") + 1
print("PM2.5 band:", pm25_band)

PM2.5 band: 5


## 2. Destination PM₂.₅ at neighbourhood representative points

This preserves the v1 destination-exposure definition so the v1/v2 difference is not confounded by changing the research metric.

In [2]:
unit_pm = units[["unit_id", "Neighbourhood_name", "Municipality_name", "geometry"]].copy()
with rasterio.open(helsinki.air_quality) as src:
    unit_pm = unit_pm.to_crs(src.crs)
    unit_pm["sample_point"] = unit_pm.geometry.representative_point()
    samples = src.sample([(p.x,p.y) for p in unit_pm["sample_point"]], indexes=pm25_band, masked=True)
    vals=[]
    for arr in samples:
        v=arr[0]
        if np.ma.is_masked(v): vals.append(np.nan)
        else:
            x=float(v); vals.append(x if x>0 else np.nan)
unit_pm["pm25_v2"] = vals

In [3]:
origin_names = units[["unit_id","Neighbourhood_name"]].rename(columns={"unit_id":"origin","Neighbourhood_name":"origin_name"})
destination_pm = unit_pm[["unit_id","pm25_v2"]].rename(columns={"unit_id":"destination","pm25_v2":"destination_pm25_v2"})

od2 = od.merge(origin_names,on="origin",how="left").merge(destination_pm,on="destination",how="left")
valid = od2.loc[od2["destination_pm25_v2"].notna()].copy()
valid["exposure_weight_v2"] = valid["trips"] * valid["destination_pm25_v2"]

origin_exposure_v2 = valid.groupby(["origin","origin_name"],as_index=False).agg(
    valid_trips=("trips","sum"), exposure_weight_v2=("exposure_weight_v2","sum")
)
origin_exposure_v2["trip_weighted_destination_pm25_v2"] = origin_exposure_v2["exposure_weight_v2"] / origin_exposure_v2["valid_trips"]

totals = od2.groupby(["origin","origin_name"],as_index=False).agg(total_trips=("trips","sum"))
origin_exposure_v2 = origin_exposure_v2.merge(totals,on=["origin","origin_name"],how="left")
origin_exposure_v2["coverage_pct_v2"] = 100*origin_exposure_v2["valid_trips"]/origin_exposure_v2["total_trips"]

display(origin_exposure_v2.sort_values("trip_weighted_destination_pm25_v2",ascending=False).head(20))

,origin,origin_name,valid_trips,exposure_weight_v2,trip_weighted_destination_pm25_v2,total_trips,coverage_pct_v2
201,917703475,Kurkimäki,34170.764165,102408.250369,2.996955,34209.760818,99.886007
137,913303213,Kyläsaari,5095.022108,15192.953515,2.981921,5096.757842,99.965944
198,917703472,Vesala,71043.827037,210468.298005,2.962514,71243.269764,99.720054
192,917701452,Puotila,48278.103319,142800.877893,2.957881,48461.940534,99.620657
194,917701455,Marjaniemi,18655.374007,54870.963643,2.941295,18725.166541,99.627280
199,917703473,Mellunmäki,88458.735574,258711.912417,2.924662,88723.821171,99.701224
200,917703474,Kivikko,48758.613456,142426.169045,2.921046,48904.688718,99.701306
193,917701453,Puotinharju,45999.973329,134298.499876,2.919534,46151.624424,99.671407
264,927095000,Rajakylä,37279.313178,108620.231282,2.913686,37402.809138,99.669822
191,917701451,Vartioharju,49243.762927,143413.737087,2.912323,49420.055368,99.643278


In [4]:
origin_exposure_v2.to_csv(OUTPUT_DIR / "mobility_weighted_destination_pm25_v2.csv", index=False)
print("Saved:", OUTPUT_DIR)

Saved: ../outputs/v2/02_mobility_weighted_destination_exposure


## 3. Interpretation for comparison

A near-identical result to v1 is expected here because Stage 02 is not a CAFEIN route-exposure operation. The purpose of v2 is to standardize the PM₂.₅ band selection and outputs while keeping the same destination-weighted metric.